# OpenMolcas SA-CASSCF-based NBRA-Workflow for Nonadiabatic Dynamics
In this tutorial, we demonstrate higher-level functions to streamline OpenMolcas calculations of the properties needed for NA-MD (nonadiabatic molecular dynamics) simulations. We will use these functions to define a workflow for NBRA (Neglect of back reaction approximation) calculations with OpenMolcas at the SA-CASSCF (State-Averaged Complete Active Space Self-Consistent Field) level of electronic structure.

The SA-CASSCF formalism provides:

Multiconfigurational description of excited states (essential for charge-transfer and multireference scenarios)
State averaging across multiple roots for balanced treatment of nonadiabatic couplings
Proper derivative couplings from the CI wavefunction structure
Improved accuracy over simpler single-reference methods (TD-DFT, TD-DFTB)
While the resulting functions are prototypes for general nonadiabatic calculations, at this point we will focus their use for NBRA-specific workflows involving:

Multiple excited-state trajectories
Time-overlap matrix construction from CI vectors
Derivative coupling extraction from finite-difference overlaps
Fewest-switches surface hopping (FSSH) and related algorithms.

Output written in results_molcas:

1. Adiabatic Hamiltoian matrix

2. Adiabatic vibronic Hamiltonian matrix

3. Atomic orbital overlap matrix

4. Time overlap matrix

It also generates molden files written in wd_molcas_itraj0 for each time steps to visualize the NTOs.

In [10]:
#!/usr/bin/env python3
"""
molcas_water_adi.py
-------------------
Nonadiabatic dynamics driver: trajectory with
OpenMolcas SA-CASSCF + full MRCI CI-state overlaps via Libra.
"""

import os
import sys
import copy
import warnings
import json
import numpy as np
from types import SimpleNamespace
from liblibra_core import Py2Cpp_int, CMATRIX  
from libra_py import units
from libra_py.packages.cp2k import methods as cp2k
import importlib
import Methods
importlib.reload(Methods)

PROJECT_DIR = "/user/someshch/Test"   # working directory
sys.path.insert(0, PROJECT_DIR)

from Methods import (
    run_molcas,
    read_molcas_orbital_info,
    read_ao_overlap,
    molcas_compute_adi,
    _infer_nstates_from_ciroot,
)

def main():
    os.chdir(PROJECT_DIR)
    labels, _ = cp2k.read_trajectory_xyz_file("pyrazine-md-aligned.xyz", 0)  # aligned nuclear steps
    natoms = len(labels)
    print(f"Atoms  : {labels}")
    print(f"Natoms : {natoms}")


    molcas_run_params = {
        "basis"      : "cc-pVDZ",
        "charge"     : 0,
        "spin"       : 1,
        "title"      : "pyrazine-md",
        "nactel"     : "6 0 0",     
        "inactive"   : 18,           
        "ras2"       : 6,          
        "ciroot"     : "3 3 1",    
        "nac_states" : None,
        "group"      : "NoSym",
        "prweight"   : 1e-6,
        "thre"       : 1e-6,
        "lshift"     : 0.5,            # Level shift for convergence
        "maxiter"    : 500,
    }
 
    params = {
        "atom_labels"              : labels,
        "timestep"                 : 0,
        "dt"                       : 1.0 * units.fs2au,
        "exe"                      : ("/projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas"),
        "molcas_run_params"        : molcas_run_params,
        "working_directory_prefix" : "wd_molcas",
        "molcas_input_prefix"      : "input_",
        "molcas_output_prefix"     : "output_",
        "nstates"                  : 3,
        "ci_coeff_thresh"          : 1e-6,  # we can reduce it
        "verbose"                  : True,  # if you need detailed diagnostic information during execution
        "do_Lowdin"                : True,
        "is_first_time"            : {0: True},

    } 
    res = "results_molcas"
    os.makedirs(res, exist_ok=True)
    full_id = Py2Cpp_int([0, 0]) 
    for i in range(50):
        print(f"\n{'='*60}")
        print(f"  Timestep {i}")
        print(f"{'='*60}")

        labels, q = cp2k.read_trajectory_xyz_file("pyrazine-md-aligned.xyz", i)
        params["timestep"] = i
 
        obj = molcas_compute_adi(q, params, full_id)

        obj.ham_adi.show_matrix(f"{res}/ham_adi_{i}.txt")
        obj.hvib_adi.show_matrix(f"{res}/hvib_adi_{i}.txt")
        obj.time_overlap_adi.real().show_matrix(f"{res}/st_adi_{i}.txt") 
        obj.overlap_adi.real().show_matrix(f"{res}/s_adi_{i}.txt")
        print(f"  Timestep {i} → saved to {res}/")
    print("\nHappy Ending (^_^).")

if __name__ == "__main__":
    main()

Atoms  : ['N', 'N', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H']
Natoms : 10

  Timestep 0
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_0_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_0_traj_0.out
[run_molcas] ✅ Completed successfully
[run_molcas] RasOrb    : wd_molcas_itraj0/input__timestep_0_traj_0.RasOrb
[DEBUG] Found 3 CI vectors with 426 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 1, -1, 0, 0), len = 6
{
    "nao": 104,
    "nmo": 104,
    "nelec": 42,
    "nocc": 18,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 18,
    "min_active": 19,
    "max_active": 24,
    "min_vir": 25,
    "max_vir": 104,
    "actual_orbital_space": [
        19,
        20,
        21,
        22,
        23,
        24
    ],


[ci_overlap_general] Total det pairs : 275100 | Skipped (|CK*CL| < 1e-06): 51468 (18.7%)
[ci_overlap_general] Total det pairs : 274576 | Skipped (|CK*CL| < 1e-06): 45177 (16.5%)
  Timestep 3 → saved to results_molcas/

  Timestep 4
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_4_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_4_traj_0.out
[run_molcas] ✅ Completed successfully
[run_molcas] RasOrb    : wd_molcas_itraj0/input__timestep_4_traj_0.RasOrb
[DEBUG] Found 3 CI vectors with 525 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
{
    "nao": 104,
    "nmo": 104,
    "nelec": 42,
    "nocc": 18,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 18,
    "min_active": 19,
    "max_active": 24,
  

[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 19481 (7.1%)
[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 8366 (3.0%)
  Timestep 7 → saved to results_molcas/

  Timestep 8
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_8_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_8_traj_0.out
[run_molcas] ✅ Completed successfully
[run_molcas] RasOrb    : wd_molcas_itraj0/input__timestep_8_traj_0.RasOrb
[DEBUG] Found 3 CI vectors with 525 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
{
    "nao": 104,
    "nmo": 104,
    "nelec": 42,
    "nocc": 18,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 18,
    "min_active": 19,
    "max_active": 24,
    "

[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 9767 (3.5%)
[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 9329 (3.4%)
  Timestep 11 → saved to results_molcas/

  Timestep 12
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_12_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_12_traj_0.out
[run_molcas] ✅ Completed successfully
[run_molcas] RasOrb    : wd_molcas_itraj0/input__timestep_12_traj_0.RasOrb
[DEBUG] Found 3 CI vectors with 525 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
{
    "nao": 104,
    "nmo": 104,
    "nelec": 42,
    "nocc": 18,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 18,
    "min_active": 19,
    "max_active": 24,
 

[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 27370 (9.9%)
[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 21765 (7.9%)
  Timestep 15 → saved to results_molcas/

  Timestep 16
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_16_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_16_traj_0.out
[run_molcas] ✅ Completed successfully
[run_molcas] RasOrb    : wd_molcas_itraj0/input__timestep_16_traj_0.RasOrb
[DEBUG] Found 3 CI vectors with 525 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
{
    "nao": 104,
    "nmo": 104,
    "nelec": 42,
    "nocc": 18,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 18,
    "min_active": 19,
    "max_active": 24,

...

[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 18919 (6.9%)
[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 8893 (3.2%)
  Timestep 31 → saved to results_molcas/

  Timestep 32
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_32_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_32_traj_0.out
[run_molcas] ✅ Completed successfully
[run_molcas] RasOrb    : wd_molcas_itraj0/input__timestep_32_traj_0.RasOrb
[DEBUG] Found 3 CI vectors with 525 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
{
    "nao": 104,
    "nmo": 104,
    "nelec": 42,
    "nocc": 18,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 18,
    "min_active": 19,
    "max_active": 24,


[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 18981 (6.9%)
[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 27299 (9.9%)
  Timestep 35 → saved to results_molcas/

  Timestep 36
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_36_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_36_traj_0.out
[run_molcas] ✅ Completed successfully
[run_molcas] RasOrb    : wd_molcas_itraj0/input__timestep_36_traj_0.RasOrb
[DEBUG] Found 3 CI vectors with 525 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
{
    "nao": 104,
    "nmo": 104,
    "nelec": 42,
    "nocc": 18,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 18,
    "min_active": 19,
    "max_active": 24,

[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 46741 (17.0%)
[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 51561 (18.7%)
  Timestep 39 → saved to results_molcas/

  Timestep 40
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_40_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_40_traj_0.out
[run_molcas] ✅ Completed successfully
[run_molcas] RasOrb    : wd_molcas_itraj0/input__timestep_40_traj_0.RasOrb
[DEBUG] Found 3 CI vectors with 525 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
{
    "nao": 104,
    "nmo": 104,
    "nelec": 42,
    "nocc": 18,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 18,
    "min_active": 19,
    "max_active": 2

[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 21293 (7.7%)
[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 20482 (7.4%)
  Timestep 43 → saved to results_molcas/

  Timestep 44
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_44_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_44_traj_0.out
[run_molcas] ✅ Completed successfully
[run_molcas] RasOrb    : wd_molcas_itraj0/input__timestep_44_traj_0.RasOrb
[DEBUG] Found 3 CI vectors with 525 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
{
    "nao": 104,
    "nmo": 104,
    "nelec": 42,
    "nocc": 18,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 18,
    "min_active": 19,
    "max_active": 24,

[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 21574 (7.8%)
[ci_overlap_general] Total det pairs : 275625 | Skipped (|CK*CL| < 1e-06): 37725 (13.7%)
  Timestep 47 → saved to results_molcas/

  Timestep 48
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_48_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_48_traj_0.out
[run_molcas] ✅ Completed successfully
[run_molcas] RasOrb    : wd_molcas_itraj0/input__timestep_48_traj_0.RasOrb
[DEBUG] Found 3 CI vectors with 525 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
{
    "nao": 104,
    "nmo": 104,
    "nelec": 42,
    "nocc": 18,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 18,
    "min_active": 19,
    "max_active": 24